# Multilingual Empathetic Dialogue System with Dynamic User-State Extraction

This project is a multilingual AI therapeutic assistant designed to support users who lack access to mental health services, offering empathy in English, Hindi, and Spanish (including code-switched phrases). At its core, the system uses Parameter-Efficient Fine-Tuning (PEFT/LoRA) on large models like Llama-3 or Qwen2.5 to provide nuanced emotional support. It also integrates Named Entity Recognition (NER) to extract user strengths, weaknesses, and goals, presenting them as interactive flashcards for self-reflection. Phase 1 is the crucial foundation where we set up the compute environment, clean and prepare our multilingual dataset, and verify that our tokenizer can handle complex language mixing. This ensures that the subsequent training and fine-tuning steps are built on a stable and high-quality data pipeline.

The three steps this notebook covers:
1. **Environment Setup**: Installing and verifying core dependencies.
2. **Dataset Preparation**: Loading and cleaning the EmpatheticDialogues dataset.
3. **Cross-Lingual Embeddings Pipeline**: Verifying multilingual tokenization capability.

## Section 1.1 — Compute Environment Setup

We begin by installing the core libraries required for our NLP pipeline. These include `transformers` for LLM management, `peft` for LoRA fine-tuning, `datasets` for data handling, `torch` for deep learning, `spacy` for NER tasks, and `flask`/`fastapi` to serve our eventual API endpoint.

In [1]:
# Install all required libraries for the project
# Using -q for quiet installation to keep the notebook clean
!pip install -q transformers peft datasets torch spacy flask fastapi scikit-learn kagglehub

After installation, we must verify that all libraries are correctly loaded and compatible. Printing the version numbers helps us confirm that the environment is ready for the deep learning tasks ahead.

In [2]:
# Import libraries and check versions to confirm successful setup
import torch
import transformers
import peft
import datasets
import spacy
import flask
import fastapi

# Print version numbers for dependency tracking
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Spacy version: {spacy.__version__}")
print(f"Flask version: {flask.__version__}")
print(f"FastAPI version: {fastapi.__version__}")

PyTorch version: 2.10.0+cpu
Transformers version: 5.0.0
PEFT version: 0.18.1
Datasets version: 4.0.0
Spacy version: 3.8.11
Flask version: 3.1.3
FastAPI version: 0.135.1


/tmp/ipykernel_13215/1631659766.py:16: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"Flask version: {flask.__version__}")


## Section 1.2 — Dataset Preparation

We are using the `EmpatheticDialogues` dataset, which captures human-like emotional nuance across various contexts. This dataset is ideal for training chatbots to respond with empathy and understanding.

In [3]:
import kagglehub

# Download latest version of the dataset from Kaggle
path = kagglehub.dataset_download("atharvjairath/empathetic-dialogues-facebook-ai")

print("Path to dataset files:", path)

100%|██████████| 3.26M/3.26M [00:00<00:00, 94.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/atharvjairath/empathetic-dialogues-facebook-ai/versions/1


In [4]:
import pandas as pd
import os
# Load the dataset using the path variable for better portability
csv_path = os.path.join(path, 'emotion-emotion_69k.csv')
raw_ds = pd.read_csv(csv_path)
raw_ds.head()

,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


To evaluate our model effectively, we split the data into training, validation, and testing sets. We aim for an 80/10/10 split using scikit-learn, ensuring we have enough data for training while keeping a representative portion for parameter tuning and final evaluation.

In [5]:
from sklearn.model_selection import train_test_split

# Perform a fresh 80/10/10 split (Train/Validation/Test)
# We first split 80% for training and 20% for testing and validation combined
train_df, temp_df = train_test_split(raw_ds, test_size=0.2, random_state=42)

# Then we split that 20% into two equal parts (10% each for validation and testing)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"New splits - Train: {len(train_df)}, Validation: {len(valid_df)}, Test: {len(test_df)}")

New splits - Train: 51708, Validation: 6464, Test: 6464


Conversational data often contains "noise" such as empty rows or redundant spaces that can skew model learning. We clean the dataset by removing null values, stripping unnecessary whitespace, and dropping exact duplicates to ensure high-quality training input.

In [6]:
import pandas as pd

def clean_data(df, split_name):
    # Convert to pandas for easier cleaning operations
    initial_count = len(df)

    # Identify the column containing the conversation text
    # In this Kaggle version, the dialogue is primarily in 'empathetic_dialogues'
    if 'empathetic_dialogues' in df.columns:
        text_col = 'empathetic_dialogues'
    elif 'utterance' in df.columns:
        text_col = 'utterance'
    else:
        text_col = df.columns[0] # Fallback to first column

    # Drop rows with null values and strip leading/trailing whitespace
    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].astype(str).str.strip()

    # Remove exact duplicate entries which add no new information
    df = df.drop_duplicates(subset=[text_col])

    final_count = len(df)
    print(f"[{split_name}] Target Column: {text_col}")
    print(f"[{split_name}] Initial: {initial_count}, Cleaned: {final_count}, Removed: {initial_count - final_count}")
    return df

# Clean all three dataset splits and print statistics
train_clean = clean_data(train_df, "Train")
valid_clean = clean_data(valid_df, "Validation")
test_clean = clean_data(test_df, "Test")

[Train] Target Column: empathetic_dialogues
[Train] Initial: 51708, Cleaned: 50766, Removed: 942
[Validation] Target Column: empathetic_dialogues
[Validation] Initial: 6464, Cleaned: 6430, Removed: 34
[Test] Target Column: empathetic_dialogues
[Test] Initial: 6464, Cleaned: 6435, Removed: 29


Noise in conversational data refers to artifacts like empty responses or repetitive text that provides no semantic value. By removing these, we prevent the model from learning biases from redundant data or crashing due to invalid input formats.

## Section 1.3 — Cross-Lingual Embeddings Pipeline

Code-switching is the practice of mixing multiple languages within a single conversation, a common trait in multilingual communities. Standard English tokenizers often fail to represent mixed-language sentences correctly, which is why we utilize XLM-RoBERTa for its superior multilingual vocabulary and cross-lingual understanding.

In [7]:
from transformers import AutoTokenizer

# Load the XLM-RoBERTa tokenizer which supports over 100 languages
# We choose XLM-RoBERTa base as it offers better cross-lingual performance than mBERT
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(f"Successfully loaded tokenizer for: {tokenizer.name_or_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Successfully loaded tokenizer for: xlm-roberta-base


We define a set of specific test sentences that blend English with Hindi and Spanish. These examples represent the natural, mixed-language style that our therapeutic assistant must be able to process to be effective in diverse linguistic environments.

In [ ]:
# Define exactly 6 test sentences featuring code-switching and a control sentence
test_sentences = [
    "I feel bahut anxious these days",                  # Hindi-English
    "Me siento very overwhelmed lately",              # Spanish-English
    "Mujhe lagta hai I am not good enough",          # Hindi-English
    "Estoy trying to be better every day",            # Spanish-English
    "I want to improve myself but kabhi kabhi it gets hard", # Hindi-English
    "I feel fine today"                               # Pure English (Control)
]
print(f"Prepared {len(test_sentences)} sentences for tokenization testing.")

Prepared 6 sentences for tokenization testing.


Next, we tokenize each sentence to observe how the text is broken down into sub-word units and numerical IDs. This step verifies that the tokenizer correctly identifies and indexes words from all target languages without losing semantic information.

In [ ]:
# Tokenize all 6 sentences and display tokens along with their internal IDs
for i, text in enumerate(test_sentences, 1):
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f"\nSentence {i}: {text}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")


Sentence 1: I feel bahut anxious these days
Tokens: ['▁I', '▁feel', '▁bahut', '▁an', 'xio', 'us', '▁these', '▁days']
Token IDs: [87, 12319, 164433, 142, 78697, 223, 6097, 13312]

Sentence 2: Me siento very overwhelmed lately
Tokens: ['▁Me', '▁siento', '▁very', '▁over', 'w', 'hel', 'med', '▁la', 'tely']
Token IDs: [1215, 141932, 4552, 645, 434, 6865, 4806, 21, 37838]

Sentence 3: Mujhe lagta hai I am not good enough
Tokens: ['▁Mujh', 'e', '▁lagt', 'a', '▁hai', '▁I', '▁am', '▁not', '▁good', '▁enough']
Token IDs: [114775, 13, 23286, 11, 1337, 87, 444, 959, 4127, 20174]

Sentence 4: Estoy trying to be better every day
Tokens: ['▁Estoy', '▁trying', '▁to', '▁be', '▁better', '▁every', '▁day']
Token IDs: [157607, 31577, 47, 186, 11522, 11907, 5155]

Sentence 5: I want to improve myself but kabhi kabhi it gets hard
Tokens: ['▁I', '▁want', '▁to', '▁improve', '▁myself', '▁but', '▁kabhi', '▁kabhi', '▁it', '▁gets', '▁hard']
Token IDs: [87, 3444, 47, 52295, 35978, 1284, 167400, 167400, 442, 62163, 

The tokenized output confirms that XLM-RoBERTa preserves the structure of code-switched sentences without defaulting to 'unknown' tokens for Hindi or Spanish words. This allows our therapeutic AI to accurately interpret the emotional state of users, regardless of language mixing.